# Spinal Disorder Classification — Stage 3B: Preprocessing Experiments

**Run only the new experiments.** This notebook is self-contained; you do **not** need to rerun the earlier Stage 3 notebook.

**Experiments**
- `E0-G1`: the original-style reference CNN with **128×128 grayscale**
- `E0-C1`: the same reference CNN with **128×128 grayscale + CLAHE**

Both follow the exact CNN training setup used by E0 (same frozen v1 split, Adam at 3e-4, weighted classes, batch size 16, checkpoint selection on validation loss). The only intended change from E0 is image preprocessing / input channels.

**Safety:** Never rebuild the split or load the 51 held-out test X-rays here. Keep dataset, private manifest, pretrained weights, image names, and checkpoint folders off public GitHub. This project is for research, not diagnosis.

This notebook will never overwrite a completed run. An existing matching result is **skipped**; an incomplete run produces a clear error for manual review.


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Imports and fixed paths

The experiment outputs go to the **same private Drive directory** as E0, E1 and E1b.

In [ ]:
from pathlib import Path
import hashlib
import json
import random
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from IPython.display import display
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    classification_report, confusion_matrix, f1_score,
)
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

DATA_ROOT = Path('/content/drive/MyDrive/224')
SPLIT_DIR = Path('/content/drive/MyDrive/Spinal_Classification_Optimization/splits/v1')
EXP_DIR = Path('/content/drive/MyDrive/Spinal_Classification_Optimization/experiments/stage3')
EXP_DIR.mkdir(parents=True, exist_ok=True)

LABEL_TO_NAME = {0: 'Normal', 1: 'Scoliosis', 2: 'Spondylolisthesis'}
CLASS_TO_LABEL = {'Normal': 0, 'Scol': 1, 'Spond': 2}

print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))
print('Experiment folder:', EXP_DIR)
if not tf.config.list_physical_devices('GPU'):
    print('NOTE: No GPU detected. Training can run on CPU but will be slower.')

## 3. Verify the frozen Stage 2 split

This verifies the saved dataset fingerprint. It does **not** generate a new split or read any test image.

In [ ]:
manifest_path = SPLIT_DIR / 'split_manifest_PRIVATE.csv'
config_path = SPLIT_DIR / 'split_config_PRIVATE.json'

assert manifest_path.is_file() and config_path.is_file(), (
    'Frozen Stage 2 split not found. Run Stage 2 first; do not generate a new split.'
)
manifest = pd.read_csv(manifest_path, keep_default_na=False)
freeze = json.loads(config_path.read_text(encoding='utf-8'))

required = {'relative_path', 'class_name', 'label', 'file_sha256', 'group_id', 'split'}
assert required <= set(manifest.columns), f'Missing manifest columns: {required - set(manifest.columns)}'
assert len(manifest) == 338 and manifest['relative_path'].is_unique
assert freeze['class_to_label'] == CLASS_TO_LABEL
assert freeze['seed'] == SEED
assert freeze['patient_independent'] is False
assert set(manifest['split']) == {'train', 'validation', 'test'}
assert manifest.groupby('group_id')['split'].nunique().max() == 1, 'Grouped images cross splits.'
assert all(int(row.label) == CLASS_TO_LABEL[row.class_name] for row in manifest.itertuples(index=False))

rows = (manifest[['relative_path', 'class_name', 'label', 'file_sha256']]
        .sort_values('relative_path')
        .to_dict(orient='records'))
fingerprint = hashlib.sha256(
    json.dumps(rows, sort_keys=True, separators=(',', ':')).encode('utf-8')
).hexdigest()
assert fingerprint == freeze['dataset_fingerprint'], 'Frozen dataset fingerprint mismatch.'

counts = pd.crosstab(manifest['class_name'], manifest['split']).reindex(
    index=list(CLASS_TO_LABEL), columns=['train', 'validation', 'test'], fill_value=0
)
display(counts.assign(total=counts.sum(axis=1)))
assert counts['train'].sum() == 235 and counts['validation'].sum() == 52 and counts['test'].sum() == 51

train_df = manifest[manifest['split'] == 'train'].sort_values('relative_path').reset_index(drop=True)
val_df = manifest[manifest['split'] == 'validation'].sort_values('relative_path').reset_index(drop=True)
# Intentional: no test_df or test image loader in this notebook.
assert set(train_df['relative_path']).isdisjoint(val_df['relative_path'])
print('Freeze verified. Development images:', len(train_df) + len(val_df))
print('Test set RESERVEd for final evaluation:', int(counts['test'].sum()))

## 4. Shared preprocessing

Use the **same** image resizing, color conversion, normalization and CLAHE implementation from Stage 3. Only train/validation images are loaded.

In [ ]:
def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def preprocess_image(path, mode, size):
    # Use EXIF orientation and identical operations for training, validation, and later inference.
    with Image.open(path) as opened:
        original = ImageOps.exif_transpose(opened).convert('RGB')
        resized = ImageOps.pad(original, (size, size), method=Image.Resampling.LANCZOS, color=(0, 0, 0))
        rgb = np.asarray(resized, dtype=np.uint8)

    if mode == 'rgb':
        image = rgb
    elif mode in {'gray', 'clahe'}:
        gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
        if mode == 'clahe':
            gray = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray)
        image = gray[..., np.newaxis]
    else:
        raise ValueError(f'Unknown preprocessing mode: {mode}')

    return image.astype(np.float32) / 255.0


def load_partition(rows, mode, size):
    X, y = [], []
    for record in rows.itertuples(index=False):
        source = DATA_ROOT / record.relative_path
        if not source.is_file() or file_sha256(source) != record.file_sha256:
            raise RuntimeError(f'Image missing or changed since freeze: {record.relative_path}')
        X.append(preprocess_image(source, mode, size))
        y.append(int(record.label))
    return np.stack(X), np.array(y, dtype=np.int64)

# Quick internal shape / range sanity check on a TRAINING image only.
sample = DATA_ROOT / train_df.iloc[0]['relative_path']
for mode in ('rgb', 'gray', 'clahe'):
    arr = preprocess_image(sample, mode=mode, size=128)
    print(mode, arr.shape, arr.dtype, (float(arr.min()), float(arr.max())))
    assert arr.shape == (128, 128, 3 if mode == 'rgb' else 1)
    assert np.isfinite(arr).all() and 0 <= arr.min() and arr.max() <= 1

## 5. Original-style CNN

Same topology as E0: three Conv2D + MaxPool blocks followed by Flatten and two Dense layers. The first convolution now accepts one channel rather than three.

In [ ]:
from tensorflow.keras import layers

def make_reference_cnn(shape):
    return tf.keras.Sequential([
        layers.Input(shape=shape),
        layers.Conv2D(32, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dense(3, activation='softmax'),
    ], name='reference_cnn')

def make_model(shape):
    model = make_reference_cnn(shape)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=['accuracy'],
    )
    return model

# Inspect architecture using the correct one-channel input.
preview = make_model((128, 128, 1))
print('Reference CNN parameters (grayscale):', preview.count_params())
del preview
tf.keras.backend.clear_session()


## 6. Choose which experiment to run

**Default: only grayscale (`E0-G1`).** Once that finishes, change `RUN_IDS` to `["E0-C1"]` and rerun **sections 6–8** for CLAHE. Do not run E0/E1/E1b again.

An already completed experiment is safely skipped. An incomplete folder is never deleted or overwritten.


In [ ]:
EXPERIMENTS = {
    'E0-G1': {'architecture': 'reference', 'mode': 'gray', 'size': 128},
    'E0-C1': {'architecture': 'reference', 'mode': 'clahe', 'size': 128},
}

RUN_IDS = ['E0-G1']   # Later change to ['E0-C1'] for CLAHE.
EPOCHS = 40
BATCH_SIZE = 16

assert RUN_IDS and len(set(RUN_IDS)) == len(RUN_IDS)
assert all(exp_id in EXPERIMENTS for exp_id in RUN_IDS)
print('Requested experiments:', RUN_IDS)

for exp_id in RUN_IDS:
    folder = EXP_DIR / exp_id
    if folder.exists():
        if (folder / 'validation_metrics.json').is_file() and (folder / 'best_model.keras').is_file():
            print(f'{exp_id}: results exist; configuration will be checked and the run skipped.')
        else:
            print(f'{exp_id}: INCOMPLETE OUTPUT FOLDER EXISTS; training will stop safely.')
    else:
        print(f'{exp_id}: ready for first training run.')


## 7. Train and save best models

Checkpoint, logs, learning curves, class recalls and configuration are saved in separate private experiment folders. **Training is done only on 235 training images** with selection based on the 52 validation images.

In [ ]:
def train_experiment(experiment_id):
    if experiment_id not in EXPERIMENTS:
        raise KeyError(f'Unknown experiment ID: {experiment_id}')

    details = dict(EXPERIMENTS[experiment_id])
    folder = EXP_DIR / experiment_id
    model_path = folder / 'best_model.keras'
    metrics_path = folder / 'validation_metrics.json'
    config_path = folder / 'configuration.json'

    # Completed runs are immutable: check their identity, then read their metrics.
    if folder.exists():
        if not (metrics_path.is_file() and model_path.is_file() and config_path.is_file()):
            raise RuntimeError(
                f'{experiment_id} has an incomplete directory: {folder}. '
                'Inspect it manually; the notebook will never overwrite it.'
            )
        existing = json.loads(config_path.read_text(encoding='utf-8'))
        expected = {
            'experiment': experiment_id, **details,
            'dataset_fingerprint': fingerprint,
            'split_version': 'v1', 'seed': SEED,
            'epochs_max': EPOCHS, 'batch_size': BATCH_SIZE,
        }
        for key, value in expected.items():
            if existing.get(key) != value:
                raise RuntimeError(
                    f'Existing {experiment_id} has incompatible {key}. '
                    'Never overwrite a previously saved experiment.'
                )
        saved_metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
        assert saved_metrics.get('test_evaluated') is False
        print(f'{experiment_id} already completed. Reusing saved metrics, no retraining.')
        return saved_metrics

    # Load ONLY train/validation images listed in the frozen manifest.
    X_train, y_train = load_partition(train_df, details['mode'], details['size'])
    X_val, y_val = load_partition(val_df, details['mode'], details['size'])
    assert X_train.shape == (235, 128, 128, 1)
    assert X_val.shape == (52, 128, 128, 1)
    assert np.array_equal(np.unique(y_train), np.arange(3))

    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)
    model = make_model(X_train.shape[1:])

    balanced_weights = compute_class_weight(
        class_weight='balanced', classes=np.arange(3), y=y_train
    )
    class_weights = {int(label): float(value)
                     for label, value in enumerate(balanced_weights)}

    settings = {
        'experiment': experiment_id,
        **details,
        'seed': SEED,
        'epochs_max': EPOCHS,
        'batch_size': BATCH_SIZE,
        'optimizer': 'Adam',
        'initial_learning_rate': 3e-4,
        'checkpoint_monitor': 'val_loss',
        'architecture_note': 'same reference CNN topology as E0; 1-channel input',
        'augmentation': 'none (matches E0)',
        'normalization': 'fixed pixel division by 255',
        'clahe_settings': (
            {'clipLimit': 2.0, 'tileGridSize': [8, 8]}
            if details['mode'] == 'clahe' else None
        ),
        'class_weights': class_weights,
        'dataset_fingerprint': fingerprint,
        'split_version': 'v1',
        'train_images': int(len(y_train)),
        'validation_images': int(len(y_val)),
        'patient_independent': False,
        'test_set_evaluated': False,
        'parameters': int(model.count_params()),
    }

    folder.mkdir(parents=True, exist_ok=False)
    config_path.write_text(json.dumps(settings, indent=2), encoding='utf-8')

    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            filepath=str(model_path), monitor='val_loss', save_best_only=True
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=9, restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5, patience=4, min_lr=1e-6
        ),
        tf.keras.callbacks.CSVLogger(str(folder / 'epoch_log.csv')),
    ]

    print(f'\\n=== {experiment_id}: {details} ===')
    start = time.perf_counter()
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        class_weight=class_weights,
        callbacks=callbacks,
        verbose=2,
        shuffle=True,
    )

    # Validation results use the best validation-LOSS checkpoint, not last weights.
    best_model = tf.keras.models.load_model(str(model_path))
    scores = best_model.predict(X_val, batch_size=BATCH_SIZE, verbose=0)
    predictions = np.argmax(scores, axis=1)

    report = classification_report(
        y_val, predictions,
        labels=[0, 1, 2],
        target_names=[LABEL_TO_NAME[i] for i in range(3)],
        output_dict=True,
        zero_division=0,
    )
    metrics = {
        'experiment': experiment_id,
        'val_accuracy': float(accuracy_score(y_val, predictions)),
        'val_macro_f1': float(
            f1_score(y_val, predictions, labels=[0, 1, 2],
                     average='macro', zero_division=0)
        ),
        'val_balanced_accuracy': float(balanced_accuracy_score(y_val, predictions)),
        'best_val_loss': float(min(history.history['val_loss'])),
        'best_epoch': int(np.argmin(history.history['val_loss']) + 1),
        'epochs_completed': int(len(history.history['loss'])),
        'runtime_seconds': float(time.perf_counter() - start),
        'per_class_recall': {
            name: float(report[name]['recall']) for name in LABEL_TO_NAME.values()
        },
        'test_evaluated': False,
    }

    # Save metrics LAST. Their existence indicates a finished run.
    pd.DataFrame(history.history).to_csv(folder / 'history.csv', index=False)
    cm = confusion_matrix(y_val, predictions, labels=[0, 1, 2])
    np.save(folder / 'confusion_matrix_validation.npy', cm)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for name, axis in [('loss', axes[0]), ('accuracy', axes[1])]:
        axis.plot(history.history[name], label='train')
        axis.plot(history.history[f'val_{name}'], label='validation')
        axis.set_title(f'{experiment_id}: {name}')
        axis.set_xlabel('Epoch')
        axis.legend()
    fig.tight_layout()
    fig.savefig(folder / 'learning_curves.png', dpi=150)
    plt.show()
    plt.close(fig)

    (folder / 'classification_report.json').write_text(
        json.dumps(report, indent=2), encoding='utf-8'
    )
    metrics_path.write_text(json.dumps(metrics, indent=2), encoding='utf-8')

    print('Validation confusion matrix (rows=true):')
    display(pd.DataFrame(
        cm, index=list(LABEL_TO_NAME.values()),
        columns=list(LABEL_TO_NAME.values())
    ))
    print('Validation metrics:', json.dumps(metrics, indent=2))
    return metrics


## 8. Run the selected experiment

If E0-G1 has already completed, this simply displays its saved metrics instead of asserting or overwriting anything.

In [ ]:
run_results = []
for experiment_id in RUN_IDS:
    run_results.append(train_experiment(experiment_id))

display(pd.DataFrame(run_results).drop(columns=['per_class_recall']).round(4))


## 9. Compare against previous experiments

Only reads saved **validation** summaries for E0, E1, E1b, E0-G1 and E0-C1. The test set stays untouched.

In [ ]:
comparison_ids = ['E0', 'E1', 'E1b', 'E0-G1', 'E0-C1']
completed = []

for experiment_id in comparison_ids:
    result_path = EXP_DIR / experiment_id / 'validation_metrics.json'
    if result_path.is_file():
        result = json.loads(result_path.read_text(encoding='utf-8'))
        if result.get('test_evaluated') is not False:
            raise RuntimeError(f'Unexpected test evaluation flag in {experiment_id}.')
        completed.append(result)

if completed:
    leaderboard = pd.DataFrame(completed).sort_values(
        ['val_macro_f1', 'best_val_loss'],
        ascending=[False, True]
    )
    display(leaderboard.drop(columns=['per_class_recall'], errors='ignore').round(4))
    print('Current validation-only leader:', leaderboard.iloc[0]['experiment'])
else:
    print('No completed experiment results found.')


## Next step

1. Run `E0-G1` first. Compare its **validation macro F1**, accuracy, loss and per-class recall to E0.
2. To run CLAHE later, change **only** `RUN_IDS = ['E0-C1']` in Section 6 and re-run Sections 6–9.
3. Preserve every run's folder and checkpoint; do not overwrite previous experiments.
4. Keep all 51 held-out test images reserved until a final architecture and preprocessing method are selected.

We have not verified patient independence. A higher validation score on 52 images is **not** reliable evidence of clinical performance. Use this as a research demonstration only.
